# Lab 5 — Error Handling
**Difficulty: Intermediate | ~25 min | Requires Lab 3**

In [1]:
# This lab uses only Python's built-in features, but this cell keeps the
# notebook runnable from a fresh kernel. It installs nothing extra.
!pip install --upgrade pip

## Basic try/except

Wrap a risky conversion like `float()` in `try`, and `except ValueError` to return a safe fallback instead of crashing.


In [2]:
# parse_price converts text to a float, catching bad input gracefully.
def parse_price(raw):
    try:
        return float(raw)  # "29.95" -> 29.95
    except ValueError:
        print(f"  Error: '{raw}' is not a valid price")
        return None  # caller can detect failure by checking for None


# --- call the function with valid and invalid inputs ---
print("--- Basic try/except ---")
print("Parsing '29.95':", parse_price("29.95"))  # valid number
print("Parsing 'abc':", parse_price("abc"))       # invalid -> handled
print("Parsing '':", parse_price(""))             # also invalid


--- Basic try/except ---
Parsing '29.95': 29.95
  Error: 'abc' is not a valid price
Parsing 'abc': None
  Error: '' is not a valid price
Parsing '': None


## Multiple specific exceptions

Stack several `except` clauses to give each error type its own handling — always catch the narrowest exception you expect.


In [3]:
# process_quantity divides a total by a quantity, catching each error type.
def process_quantity(quantity, total_price):
    try:
        result = total_price / quantity
        print(f"  Processing quantity {quantity}: result = {result}")
    except TypeError:  # e.g. dividing by a string
        print(f"  Error: Cannot divide string by int")
        return None
    except ZeroDivisionError:  # quantity was 0
        print(f"  Error: division by zero")
        return None
    return result


# --- call with different inputs to exercise every branch ---
print("--- Multiple Specific Exceptions ---")
print(process_quantity(5, 100))        # works normally
print(process_quantity("five", 100))  # TypeError path
print(process_quantity(0, 100))        # ZeroDivisionError path
print(process_quantity(3.5, 112))      # float is fine


--- Multiple Specific Exceptions ---
  Processing quantity 5: result = 20.0
20.0
  Error: Cannot divide string by int
None
  Error: division by zero
None
  Processing quantity 3.5: result = 32.0
32.0


## try/finally cleanup

`finally` runs whether the `try` succeeds or raises, so it's the right place for cleanup like closing files.


In [4]:
# finally always runs, so cleanup happens whether an error occurred or not.
print("--- try/finally Cleanup ---")

for qty in [5, "bad"]:
    resource = "file_sim.txt"

    try:
        print(f"OPENED: resource {resource}")
        if isinstance(qty, int):
            print(f"  Processing {resource} ...")
        else:
            raise RuntimeError("simulated crash!")  # force an error
    except RuntimeError as e:
        print(f"  Error: {e}")
    finally:
        print(f"CLOSED: resource {resource}")  # runs in BOTH cases

    print("Processing continued after error.\n")


--- try/finally Cleanup ---
OPENED: resource file_sim.txt
  Processing file_sim.txt ...
CLOSED: resource file_sim.txt
Processing continued after error.

OPENED: resource file_sim.txt
  Error: simulated crash!
CLOSED: resource file_sim.txt
Processing continued after error.



## Custom exception: InvalidOrderError

Subclass `Exception` to name a domain error and carry extra fields, separating business-rule failures from runtime errors.


In [5]:
# A custom exception carries extra fields to aid diagnosis.
class InvalidOrderError(Exception):
    def __init__(self, field, value, message):
        super().__init__(message)  # keep the normal message behaviour
        self.field = field  # which field was invalid
        self.value = value  # what value was supplied


# --- raise and catch the custom exception ---
print("--- Custom Exception: InvalidOrderError ---")

try:
    qty = 0
    if qty < 1:
        raise InvalidOrderError("quantity", qty,
            f"Order for 'Shirt' must have quantity >= 1, got {qty}")
except InvalidOrderError as e:  # catch our own type specifically
    print(f"{type(e).__name__}: {e}")


--- Custom Exception: InvalidOrderError ---
InvalidOrderError: Order for 'Shirt' must have quantity >= 1, got 0


## Re-raising errors

`raise` inside an `except` (bare, no arguments) logs or inspects the error, then lets it keep propagating with its traceback.


In [6]:
# Log the problem, then re-raise so the caller still sees the error.
def divide_for_report(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print(f"ERROR (logged): division by zero")
        raise  # bare raise -> send the SAME exception onward


# --- outer handler catches the re-raised exception ---
print("--- Re-raising Errors ---")

try:
    divide_for_report(100, 0)
except ZeroDivisionError as e:  # caught again by the outer block
    print(f"Traceback caught: {e}")


--- Re-raising Errors ---
ERROR (logged): division by zero
Traceback caught: division by zero


## Robust order validation

Collect *all* validation errors before raising one rich error, so the caller sees every problem in a single pass.


In [7]:
# Robust validation: collect every problem, then raise one rich error.
def validate_order(order):
    errors = []

    # 1. check product name -- must be a non-empty string.
    if not isinstance(order.get("product"), str) or not order["product"].strip():
        errors.append("requires a valid product name")

    # 2. check price -- must be a positive number (even if given as text).
    try:
        price = float(order.get("price", 0))
        if price <= 0:
            errors.append(
                f"Price for '{order.get('product')}' must be > 0, got {price:.2f}")
    except (TypeError, ValueError):
        errors.append(f"Price '{order.get('price')}' is not a valid number")

    # 3. check quantity -- must be a positive whole number.
    try:
        qty = int(order.get("quantity", 0))
        if qty < 1:
            errors.append(
                f"Order for '{order.get('product')}' must have quantity >= 1, got {qty}")
    except (TypeError, ValueError):
        errors.append(
            f"Quantity '{order.get('quantity')}' is not a valid integer")

    # 4. raise if any problem was found.
    if errors:
        raise InvalidOrderError("order", str(order), "; ".join(errors))

    return True


# --- test data: one valid order and three with different problems ---
orders = [
    {"product": "Shirt",  "price": 25.00, "quantity": 2},   # valid
    {"product": "Hat",    "price": 15.00, "quantity": -2},  # bad qty
    {"product": "Scarf",  "price": -15.00, "quantity": 1},  # bad price
    {"product": "",       "price": 40.00, "quantity": 1},   # empty name
]


# --- validate each order and report results ---
print("--- Robust Order Validation ---")

valid = 0
for i, order in enumerate(orders, 1):
    try:
        validate_order(order)
        print(f"  Order {i} PASSED")
        valid += 1
    except InvalidOrderError as e:
        print(f"  Order {i} FAILED: {e}")

print(f"Processing complete: {valid} valid, {len(orders) - valid} invalid.")


--- Robust Order Validation ---
  Order 1 PASSED
  Order 2 FAILED: Order for 'Hat' must have quantity >= 1, got -2
  Order 3 FAILED: Price for 'Scarf' must be > 0, got -15.00
  Order 4 FAILED: requires a valid product name
Processing complete: 1 valid, 3 invalid.


## Optional Exercise

Extend `validate_order` to also validate an optional `"discount"` field:

- If `"discount"` is present, it must be a number between `0` and `50` (inclusive).
- If it's missing, default to `0` (no discount).
- If it's out of range, add an error to the errors list: `"Discount must be 0-50, got {value}"`.

Test your updated function by adding a fifth order with `"discount": 75` and confirm the error message appears in the output.